# 🍷 Wine Quality — Multi-Model Experiments + DagsHub

Cada modelo é logado como um **run separado** no DagsHub/MLflow para
comparação visual de métricas e artefatos no UI.

## Fluxo
```
Dataset → Feature Eng → Split → SMOTENC → Loop de Experimentos
                                               ↓ (para cada modelo)
                                           MLflow Run no DagsHub
                                           Params + Metrics + Confusion Matrix
                                           Model Registry
```

## Modelos comparados
| # | Modelo | Estratégia de desbalanceamento |
|---|---|---|
| 1 | RandomForest | `class_weight='balanced'` + SMOTENC |
| 2 | ExtraTreesClassifier | `class_weight='balanced'` + SMOTENC |
| 3 | HistGradientBoosting | `class_weight` nativo + SMOTENC |
| 4 | XGBoost | `sample_weight` calculado + SMOTENC |
| 5 | LightGBM | `class_weight` dict + SMOTENC |
| 6 | StackingClassifier | ensemble dos melhores |

## Estratégia de Classes Raras
- Classes 9 (5 amostras) e 3 (30 amostras) são fusionadas antes do SMOTE:
  `9 → 8` e `3 → 4` (configurável via `MERGE_EXTREME_CLASSES`)
- SMOTENC mais agressivo: alvo de 800–1000 amostras nas classes minoritárias

## 0. Ambiente e Conexão DagsHub

In [ ]:
from dotenv import load_dotenv
load_dotenv('../.env', override=True)

import os
DAGSHUB_USERNAME  = os.environ['DAGSHUB_USERNAME']
DAGSHUB_REPO_NAME = os.environ['DAGSHUB_REPO_NAME']
DAGSHUB_TOKEN     = os.environ['DAGSHUB_TOKEN']
MLFLOW_EXPERIMENT = os.getenv('MLFLOW_EXPERIMENT', 'wine-quality')
REGISTERED_MODEL  = os.getenv('MLFLOW_MODEL_NAME',  'wine-quality')

# ─── Configurações do experimento ────────────────────────────────────────
MERGE_EXTREME_CLASSES = True   # True: fusiona 9→8 e 3→4 antes do SMOTE
SMOTE_K_NEIGHBORS     = 5
RANDOM_STATE          = 42
CV_FOLDS              = 5
N_ITER_SEARCH         = 25    # aumentar para busca mais ampla (mais lento)

print(f'Experimento     : {MLFLOW_EXPERIMENT}')
print(f'Merge extremas  : {MERGE_EXTREME_CLASSES}')

In [ ]:
import warnings, json, time, os
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')          # headless para salvar sem display
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn
from sklearn.model_selection   import train_test_split, RandomizedSearchCV
from sklearn.ensemble          import (
    RandomForestClassifier, ExtraTreesClassifier,
    HistGradientBoostingClassifier, StackingClassifier
)
from sklearn.linear_model      import LogisticRegression
from sklearn.pipeline          import Pipeline
from sklearn.compose           import ColumnTransformer
from sklearn.preprocessing     import StandardScaler, OneHotEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics           import (
    classification_report, accuracy_score,
    f1_score, ConfusionMatrixDisplay,
    precision_recall_fscore_support
)
from scipy.stats import randint, uniform

# Boosting
from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier

# Balanceamento
from imblearn.over_sampling  import SMOTENC, RandomOverSampler
from imblearn.pipeline       import Pipeline as ImbPipeline

# MLflow / DagsHub
import mlflow
import mlflow.sklearn
from mlflow                  import MlflowClient
from mlflow.models.signature import infer_signature
import dagshub

os.makedirs('../reports', exist_ok=True)
print('✅  Imports OK')

In [ ]:
dagshub.init(
    repo_owner=DAGSHUB_USERNAME,
    repo_name=DAGSHUB_REPO_NAME,
    mlflow=True,
)
os.environ['MLFLOW_TRACKING_USERNAME'] = DAGSHUB_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = DAGSHUB_TOKEN

mlflow.set_experiment(MLFLOW_EXPERIMENT)
print('Tracking URI:', mlflow.get_tracking_uri())

## 1. EDA — Desbalanceamento

In [ ]:
df = pd.read_csv('../data/wine_quality_merged.csv')
print(f'Shape: {df.shape}')

vc = df['quality'].value_counts().sort_index()
total = len(df)

print('\nDistribuição de classes:')
for q, c in vc.items():
    bar = '█' * int(c / total * 100)
    print(f'  {q}: {c:5d} ({c/total*100:4.1f}%) {bar}')

print(f'\nRazão máx/mín: {vc.max()/vc.min():.0f}x  ← problema central')
print(f'Classes 3 e 9 no teste (~20%): ~{int(vc[3]*0.2)} e ~{int(vc[9]*0.2)} amostras')

## 2. Feature Engineering + Fusão de Classes Extremas

**Por que fusionar 9→8 e 3→4?**

Classe 9 tem apenas **5 amostras totais** → ~1 no teste. É impossível aprender um padrão
com 4 exemplos de treino. Classe 3 tem 30 amostras (~6 no teste) — ainda crítico.

A fusão é **semanticamente coerente**: nota 9 e 8 ambas representam "vinho excepcional";
nota 3 e 4 ambas representam "qualidade baixa". A fronteira de decisão muda minimamente
mas a aprendizagem melhora substancialmente.

Configure `MERGE_EXTREME_CLASSES = False` para desativar e comparar o impacto.

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Features derivadas — aplicar ANTES e DEPOIS do SMOTE."""
    df = df.copy()
    df['alcohol_sulphates']  = df['alcohol'] * df['sulphates']
    df['acid_ratio']         = df['fixed acidity'] / (df['volatile acidity'] + 1e-6)
    df['so2_ratio']          = df['free sulfur dioxide'] / (df['total sulfur dioxide'] + 1e-6)
    df['density_alcohol']    = df['density'] / (df['alcohol'] + 1e-6)   # nova feature
    df['sugar_alcohol_ratio']= df['residual sugar'] / (df['alcohol'] + 1e-6)  # nova feature
    for col in ['residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide']:
        df[f'log_{col.replace(" ","_")}'] = np.log1p(df[col])
    return df


df_feat = add_features(df)

# Fusão de classes extremas
if MERGE_EXTREME_CLASSES:
    df_feat['quality'] = df_feat['quality'].replace({9: 8, 3: 4})
    print('Classes após fusão:', sorted(df_feat['quality'].unique()))
    print(df_feat['quality'].value_counts().sort_index().to_string())
else:
    print('Sem fusão de classes.')

from sklearn.preprocessing import LabelEncoder

X = df_feat.drop(columns=['quality'])
y_orig = df_feat['quality']

# --- NOVO: Mapeando as classes para 0, 1, 2, 3, 4 (Exigência do XGBoost) ---
le = LabelEncoder()
y = pd.Series(le.fit_transform(y_orig), name='quality', index=y_orig.index)

print(f"\nMapeamento de classes (LabelEncoder):")
for orig, enc in zip(le.classes_, le.transform(le.classes_)):
    print(f"  Qualidade original {orig} -> Nova classe {enc}")
# -------------------------------------------------------------------------

CAT_COL   = 'type'
CAT_INDEX = list(X.columns).index(CAT_COL)
print(f'\nFeatures totais: {X.shape[1]}  |  CAT_INDEX: {CAT_INDEX}')

## 3. Split Treino / Teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Treino: {X_train.shape[0]} | Teste: {X_test.shape[0]}')
print('\nDistribuição no treino:')
vc_train = y_train.value_counts().sort_index()
for q, c in vc_train.items():
    print(f'  cls {q}: {c:5d}')

## 4. SMOTENC — Estratégia Agressiva

Com `MERGE_EXTREME_CLASSES=True`, as classes resultantes são {4, 5, 6, 7, 8}.
Classe 4 (agora 246 amostras) e 8 (198 amostras) recebem oversample agressivo.

**Targets SMOTENC:**
- Sem fusão: 3→1000, 4→800, 8→700, 9→500  
- Com fusão:  4→1000, 8→800

In [ ]:
# ── Passo 1: Random Oversample para classes com < 15 amostras ─────────────
current = y_train.value_counts()
ros_strategy = {cls: 25 for cls, cnt in current.items() if cnt < 15}
if ros_strategy:
    print('RandomOverSampler strategy:', ros_strategy)
    ros = RandomOverSampler(sampling_strategy=ros_strategy, random_state=RANDOM_STATE)
    X_ros, y_ros = ros.fit_resample(X_train, y_train)
else:
    X_ros, y_ros = X_train.copy(), y_train.copy()

# ── Passo 2: SMOTENC agressivo ───────────────────────────────────────────
current2 = pd.Series(y_ros).value_counts()

# --- MODIFICADO: Traduzindo os alvos originais para o formato codificado ---
if MERGE_EXTREME_CLASSES:
    smote_targets_orig = {4: 1000, 8: 800}
else:
    smote_targets_orig = {3: 1000, 4: 800, 8: 700, 9: 500}

# Usa o `le` treinado na célula 9 para adaptar as chaves do dicionário
smote_targets = {le.transform([k])[0]: v for k, v in smote_targets_orig.items()}

# Filtra classes que já superaram o alvo
smote_strategy = {k: v for k, v in smote_targets.items()
                  if current2.get(k, 0) < v}
print('SMOTENC strategy:', smote_strategy)

smotenc = SMOTENC(
    categorical_features=[CAT_INDEX],
    sampling_strategy=smote_strategy,
    k_neighbors=SMOTE_K_NEIGHBORS,
    random_state=RANDOM_STATE,
)
X_res, y_res = smotenc.fit_resample(X_ros, y_ros)

print(f'\nTreino original  : {X_train.shape[0]:5d} amostras')
print(f'Treino balanceado: {len(X_res):5d} amostras')

# Reconstrói DataFrame com dtypes corretos
X_train_bal = pd.DataFrame(X_res, columns=X.columns)
y_train_bal = pd.Series(y_res, name='quality').astype(int)
for col in X_train_bal.columns:
    if col != CAT_COL:
        X_train_bal[col] = X_train_bal[col].astype(float)

# Validação: tipo categórico
type_unique = set(X_train_bal[CAT_COL].unique())
assert type_unique <= {'red', 'white'}, f'Valor inválido em type: {type_unique}'
print('✅  Categorias válidas:', type_unique)

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
vc_before = y_train.value_counts().sort_index()
vc_after  = y_train_bal.value_counts().sort_index()
for ax, vc, title, color in [
    (axes[0], vc_before, 'Antes do SMOTENC', 'salmon'),
    (axes[1], vc_after,  'Após SMOTENC',     'mediumseagreen'),
]:
    ax.bar(vc.index.astype(str), vc.values, color=color, edgecolor='white')
    ax.set_title(title); ax.set_xlabel('Quality')
    for x, v in zip(vc.index, vc.values):
        ax.text(str(x), v+5, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../reports/smote_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(y_train_bal.value_counts().sort_index().to_string())

## 5. Preprocessor Compartilhado

In [ ]:
num_cols = [c for c in X.columns if c != CAT_COL]
cat_cols = [CAT_COL]

preprocessor = ColumnTransformer([
    ('num', StandardScaler(),                     num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
], remainder='drop')

print(f'Features numéricas: {len(num_cols)}')
print(f'Features categóricas: {cat_cols}')

## 6. Registro de Experimentos

Cada entrada no dicionário `EXPERIMENTS` define:
- O estimador base
- O espaço de busca de hiperparâmetros
- Se usa `sample_weight` no fit (XGBoost/LightGBM)

O loop da seção seguinte itera sobre esse dicionário e cria
**um run MLflow por modelo** → comparação direta no DagsHub.

In [ ]:
# Classes únicas para configurar n_classes nos modelos
CLASSES       = sorted(y.unique())
N_CLASSES     = len(CLASSES)
CLASS_MAP     = {c: i for i, c in enumerate(CLASSES)}   # para XGBoost

# Pesos por classe (para modelos que aceitam sample_weight)
def get_sample_weights(y_series):
    return compute_sample_weight('balanced', y_series)

print(f'Classes: {CLASSES}  ({N_CLASSES} classes)')

EXPERIMENTS = {

    # ── 1. RandomForest ──────────────────────────────────────────────────
    'RandomForest': {
        'estimator': RandomForestClassifier(
            class_weight='balanced_subsample',
            random_state=RANDOM_STATE, n_jobs=-1
        ),
        'params': {
            'clf__n_estimators':     randint(300, 700),
            'clf__max_depth':        [None, 15, 25, 35],
            'clf__min_samples_leaf': randint(1, 6),
            'clf__max_features':     ['sqrt', 'log2', 0.4],
        },
        'use_sample_weight': False,
    },

    # ── 2. ExtraTreesClassifier ──────────────────────────────────────────
    'ExtraTrees': {
        'estimator': ExtraTreesClassifier(
            class_weight='balanced_subsample',
            random_state=RANDOM_STATE, n_jobs=-1
        ),
        'params': {
            'clf__n_estimators':     randint(300, 700),
            'clf__max_depth':        [None, 15, 25, 35],
            'clf__min_samples_leaf': randint(1, 6),
            'clf__max_features':     ['sqrt', 'log2', 0.4],
        },
        'use_sample_weight': False,
    },

    # ── 3. HistGradientBoosting (sklearn nativo, rápido) ─────────────────
    'HistGradientBoosting': {
        'estimator': HistGradientBoostingClassifier(
            class_weight='balanced',
            random_state=RANDOM_STATE,
            early_stopping=True, validation_fraction=0.1
        ),
        'params': {
            'clf__max_iter':         randint(200, 600),
            'clf__learning_rate':    uniform(0.03, 0.15),
            'clf__max_depth':        randint(4, 12),
            'clf__min_samples_leaf': randint(10, 50),
            'clf__l2_regularization': uniform(0.0, 1.0),
        },
        'use_sample_weight': False,
    },

    # ── 4. XGBoost ────────────────────────────────────────────────────────
    # XGBoost não tem class_weight nativo para multiclasse → usa sample_weight
    'XGBoost': {
        'estimator': XGBClassifier(
            objective='multi:softprob',
            num_class=N_CLASSES,
            eval_metric='mlogloss',
            use_label_encoder=False,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=0,
        ),
        'params': {
            'clf__n_estimators':  randint(200, 600),
            'clf__learning_rate': uniform(0.03, 0.15),
            'clf__max_depth':     randint(4, 10),
            'clf__subsample':     uniform(0.6, 0.4),
            'clf__colsample_bytree': uniform(0.5, 0.5),
            'clf__reg_alpha':     uniform(0, 1.0),
            'clf__reg_lambda':    uniform(0.5, 2.0),
        },
        'use_sample_weight': True,   # passa balanced weights no fit
    },

    # ── 5. LightGBM ───────────────────────────────────────────────────────
    'LightGBM': {
        'estimator': LGBMClassifier(
            class_weight='balanced',
            objective='multiclass',
            num_class=N_CLASSES,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        ),
        'params': {
            'clf__n_estimators':   randint(200, 600),
            'clf__learning_rate':  uniform(0.03, 0.15),
            'clf__num_leaves':     randint(20, 100),
            'clf__max_depth':      randint(4, 12),
            'clf__min_child_samples': randint(5, 30),
            'clf__reg_alpha':      uniform(0, 1.0),
            'clf__reg_lambda':     uniform(0, 1.0),
            'clf__subsample':      uniform(0.6, 0.4),
        },
        'use_sample_weight': False,
    },
}

## 7. Loop de Experimentos → DagsHub

Para cada modelo:
1. `RandomizedSearchCV` com 5-fold CV no treino balanceado
2. Avalia no teste **original** (sem SMOTE)
3. Loga tudo em um `mlflow.start_run()` separado
4. Salva confusion matrix como artefato
5. Registra o pipeline no Model Registry

> 🕐 Tempo estimado: ~15–30 min dependendo da máquina

In [ ]:
def log_confusion_matrix(y_true, y_pred, model_name: str, run_id: str) -> str:
    """Salva a confusion matrix como PNG e retorna o path."""
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred, ax=ax, cmap='Blues', colorbar=False
    )
    ax.set_title(f'Confusion Matrix — {model_name}')
    plt.tight_layout()
    path = f'../reports/cm_{model_name.lower().replace(" ","_")}_{run_id[:8]}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    return path


def run_experiment(
    name: str,
    estimator,
    param_dist: dict,
    use_sample_weight: bool,
    X_tr, y_tr,
    X_te, y_te,
    smote_strategy: dict,
) -> dict:
    """
    Treina um modelo com RandomizedSearchCV, avalia no teste
    e loga tudo como um run MLflow independente no DagsHub.
    Retorna dict com métricas resumidas.
    """

    print(f'\n{"─"*60}')
    print(f'  Iniciando experimento: {name}')
    print(f'{"─"*60}')

    # ── Pipeline ─────────────────────────────────────────────────────
    pipe = Pipeline([
        ('pre', preprocessor),
        ('clf', estimator),
    ])

    # ── RandomizedSearch ─────────────────────────────────────────────
    search = RandomizedSearchCV(
        pipe, param_dist,
        n_iter=N_ITER_SEARCH,
        cv=CV_FOLDS,
        scoring='f1_macro',    # métrica de busca focada em classes raras
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=0,
        refit=True,
        error_score='raise',
    )

    t0 = time.perf_counter()

    if use_sample_weight:
        # XGBoost: pass sample_weight no fit_params
        sw = get_sample_weights(y_tr)
        # Nota: nome do step do Pipeline é 'clf'
        search.fit(X_tr, y_tr, clf__sample_weight=sw)
    else:
        search.fit(X_tr, y_tr)

    elapsed_train = round(time.perf_counter() - t0, 1)

    best_model = search.best_estimator_

    # ── Avaliação no teste original ───────────────────────────────────
    y_pred = best_model.predict(X_te)

    acc     = round(accuracy_score(y_te, y_pred), 4)
    f1_w    = round(f1_score(y_te, y_pred, average='weighted', zero_division=0), 4)
    f1_mac  = round(f1_score(y_te, y_pred, average='macro',    zero_division=0), 4)
    cv_best = round(search.best_score_, 4)

    prec, rec, f1c, _ = precision_recall_fscore_support(
        y_te, y_pred,
        labels=sorted(y_te.unique()),
        zero_division=0
    )

    print(f'  CV F1-macro (treino) : {cv_best}')
    print(f'  Accuracy (teste)     : {acc}')
    print(f'  F1 macro (teste)     : {f1_mac}')
    print(f'  F1 weighted (teste)  : {f1_w}')
    print(f'  Tempo de treino      : {elapsed_train}s')

    # ── MLflow Run ────────────────────────────────────────────────────
    with mlflow.start_run(run_name=name) as run:
        run_id = run.info.run_id

        # Tags (identificam o run no DagsHub)
        # Ensure keys are JSON-serializable (numpy.int64 -> int)
        smote_strategy_serializable = {int(k): int(v) for k, v in smote_strategy.items()}
        mlflow.set_tags({
            'model_name'        : name,
            'balancing'         : 'SMOTENC',
            'merge_extremes'    : str(MERGE_EXTREME_CLASSES),
            'smote_strategy'    : json.dumps(smote_strategy_serializable),
            'use_sample_weight' : str(use_sample_weight),
            'cv_scoring'        : 'f1_macro',
        })

        # Params do melhor modelo
        best_params_flat = {
            k.replace('clf__', ''): str(v)
            for k, v in search.best_params_.items()
        }
        mlflow.log_params({
            **best_params_flat,
            'cv_best_f1_macro': cv_best,
            'train_size_orig'  : X_train.shape[0],
            'train_size_bal'   : X_tr.shape[0],
            'test_size'        : X_te.shape[0],
            'n_features'       : X.shape[1],
            'n_classes'        : N_CLASSES,
            'train_elapsed_s'  : elapsed_train,
        })

        # Métricas globais
        mlflow.log_metrics({
            'test_accuracy'   : acc,
            'test_f1_weighted': f1_w,
            'test_f1_macro'   : f1_mac,
        })

        # Métricas por classe
        for cls, p, r, f in zip(sorted(y_te.unique()), prec, rec, f1c):
            mlflow.log_metrics({
                f'precision_cls{cls}': round(float(p), 4),
                f'recall_cls{cls}'   : round(float(r), 4),
                f'f1_cls{cls}'       : round(float(f), 4),
            })

        # Relatório JSON
        report = classification_report(y_te, y_pred, output_dict=True, zero_division=0)
        rp_path = f'../reports/report_{name.lower()}.json'
        with open(rp_path, 'w') as fp:
            json.dump(report, fp, indent=2)
        mlflow.log_artifact(rp_path, artifact_path='reports')

        # Confusion Matrix
        cm_path = log_confusion_matrix(y_te, y_pred, name, run_id)
        mlflow.log_artifact(cm_path, artifact_path='reports')

        # Artifact SMOTE comparison (apenas no primeiro run)
        if os.path.exists('../reports/smote_comparison.png'):
            mlflow.log_artifact('../reports/smote_comparison.png', 'reports')

        # Modelo serializado (não bloqueante) — captura falhas de upload
        sig = infer_signature(X_te, y_pred)
        try:
            mlflow.sklearn.log_model(
                sk_model=best_model,
                name='model',
                signature=sig,
                input_example=X_te.iloc[:3],
                registered_model_name=REGISTERED_MODEL,
                await_registration_for=0,
            )
        except Exception as e:
            # registra o erro como tag e continua o loop
            try:
                mlflow.set_tag('model_log_error', str(e))
            except Exception:
                pass
            print(f'⚠️  Model logging failed: {e} — continuing without blocking.')

        print(f'  ✅  Run logada: {run_id}')

    return {
        'model'    : name,
        'run_id'   : run_id,
        'accuracy' : acc,
        'f1_macro' : f1_mac,
        'f1_weighted': f1_w,
        'cv_f1_macro': cv_best,
        'best_estimator': best_model,
    }

print('✅  Helpers definidos')

In [ ]:
# ── Loop principal ────────────────────────────────────────────────────────
# Cada modelo gera um run separado no DagsHub → comparação direta no UI

all_results = []

for model_name, config in EXPERIMENTS.items():
    result = run_experiment(
        name=model_name,
        estimator=config['estimator'],
        param_dist=config['params'],
        use_sample_weight=config['use_sample_weight'],
        X_tr=X_train_bal,
        y_tr=y_train_bal,
        X_te=X_test,
        y_te=y_test,
        smote_strategy=smote_strategy,
    )
    all_results.append(result)

print('\n' + '='*60)
print('EXPERIMENTOS CONCLUÍDOS')
print('='*60)

## 8. Comparação Local dos Resultados

In [ ]:
df_results = pd.DataFrame([
    {k: v for k, v in r.items() if k != 'best_estimator'}
    for r in all_results
]).sort_values('f1_macro', ascending=False)

print('\n📊  RANKING DOS MODELOS (ordenado por F1 macro):')
print(df_results[['model','accuracy','f1_macro','f1_weighted','cv_f1_macro']]
      .to_string(index=False))

# Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = ['accuracy', 'f1_macro', 'f1_weighted']
titles  = ['Accuracy', 'F1 Macro', 'F1 Weighted']
colors  = ['steelblue', 'salmon', 'mediumseagreen']

for ax, metric, title, color in zip(axes, metrics, titles, colors):
    df_sorted = df_results.sort_values(metric, ascending=True)
    bars = ax.barh(df_sorted['model'], df_sorted[metric], color=color, edgecolor='white')
    ax.set_title(title)
    ax.set_xlim(0, 1)
    for bar, val in zip(bars, df_sorted[metric]):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)

plt.suptitle('Comparação de Modelos (conjunto de teste)', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/model_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# F1 por classe para cada modelo
fig, ax = plt.subplots(figsize=(14, 6))

classes_in_test = sorted(y_test.unique())
x = np.arange(len(classes_in_test))
width = 0.8 / len(all_results)
palette = plt.cm.tab10(np.linspace(0, 1, len(all_results)))

for i, result in enumerate(all_results):
    model = result['best_estimator']
    y_pred = model.predict(X_test)
    _, _, f1c, _ = precision_recall_fscore_support(
        y_test, y_pred,
        labels=classes_in_test,
        zero_division=0
    )
    offset = (i - len(all_results)/2 + 0.5) * width
    ax.bar(x + offset, f1c, width, label=result['model'],
           color=palette[i], edgecolor='white', alpha=0.85)

ax.set_xlabel('Classe de Qualidade')
ax.set_ylabel('F1-score')
ax.set_title('F1 por Classe — Todos os Modelos')
ax.set_xticks(x)
ax.set_xticklabels([str(c) for c in classes_in_test])
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../reports/f1_per_class_all_models.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Stacking Ensemble (opcional)

Usa os melhores modelos base como estimadores e um `LogisticRegression`
como meta-estimador. Logado como run adicional no DagsHub.

In [ ]:
# Seleciona os 3 melhores modelos individuais para o stack
top3 = sorted(all_results, key=lambda r: r['f1_macro'], reverse=True)[:3]
print('Top 3 para stacking:')
for r in top3:
    print(f"  {r['model']:25s}  F1_macro={r['f1_macro']}")

# Estimadores base (já incluem o preprocessor no pipeline)
base_estimators = [(r['model'], r['best_estimator']) for r in top3]

stacking = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(
        max_iter=1000, C=1.0,
        class_weight='balanced',
        random_state=RANDOM_STATE
    ),
    cv=CV_FOLDS,
    n_jobs=-1,
    passthrough=False,
)

print('\nTreinando StackingClassifier...')
t0 = time.perf_counter()
stacking.fit(X_train_bal, y_train_bal)
elapsed = round(time.perf_counter() - t0, 1)

y_pred_stack = stacking.predict(X_test)
acc_s   = round(accuracy_score(y_test, y_pred_stack), 4)
f1_m_s  = round(f1_score(y_test, y_pred_stack, average='macro',    zero_division=0), 4)
f1_w_s  = round(f1_score(y_test, y_pred_stack, average='weighted', zero_division=0), 4)

print(f'Stacking — Accuracy: {acc_s} | F1 macro: {f1_m_s} | F1 weighted: {f1_w_s}')
print(classification_report(y_test, y_pred_stack, zero_division=0))

# Log do stacking como run separado
with mlflow.start_run(run_name='StackingEnsemble') as run:
    run_id_stack = run.info.run_id
    mlflow.set_tags({
        'model_name' : 'StackingEnsemble',
        'base_models': ', '.join([r['model'] for r in top3]),
        'balancing'  : 'SMOTENC',
    })
    mlflow.log_params({
        'base_models'     : str([r['model'] for r in top3]),
        'meta_estimator'  : 'LogisticRegression',
        'train_elapsed_s' : elapsed,
        'n_features'      : X.shape[1],
        'train_size_bal'  : X_train_bal.shape[0],
    })
    mlflow.log_metrics({
        'test_accuracy'   : acc_s,
        'test_f1_macro'   : f1_m_s,
        'test_f1_weighted': f1_w_s,
    })
    prec_s, rec_s, f1c_s, _ = precision_recall_fscore_support(
        y_test, y_pred_stack, labels=sorted(y_test.unique()), zero_division=0
    )
    for cls, p, r, f in zip(sorted(y_test.unique()), prec_s, rec_s, f1c_s):
        mlflow.log_metrics({f'f1_cls{cls}': round(float(f), 4)})

    cm_path_s = log_confusion_matrix(y_test, y_pred_stack, 'StackingEnsemble', run_id_stack)
    mlflow.log_artifact(cm_path_s, artifact_path='reports')

    sig_s = infer_signature(X_test, y_pred_stack)
    mlflow.sklearn.log_model(
        sk_model=stacking,
        artifact_path='model',
        signature=sig_s,
        input_example=X_test.iloc[:3],
        registered_model_name=REGISTERED_MODEL,
    )
    print(f'✅  Stacking logado: {run_id_stack}')

all_results.append({
    'model': 'StackingEnsemble', 'run_id': run_id_stack,
    'accuracy': acc_s, 'f1_macro': f1_m_s, 'f1_weighted': f1_w_s,
    'cv_f1_macro': None, 'best_estimator': stacking,
})

## 10. Promoção do Melhor Modelo para Production

Após comparar visualmente no DagsHub, promova manualmente ou execute esta célula
para promover o melhor por F1 macro automaticamente.

In [ ]:
# Identifica melhor run por F1 macro
best = max(all_results, key=lambda r: r['f1_macro'])
best_run_id = best['run_id']
print(f'Melhor modelo   : {best["model"]}')
print(f'F1 macro (teste): {best["f1_macro"]}')
print(f'Run ID          : {best_run_id}')

# Busca a versão do modelo correspondente ao run
client = MlflowClient()
time.sleep(3)    # aguarda propagação no DagsHub

all_versions = client.search_model_versions(f"name='{REGISTERED_MODEL}'")
matching = [v for v in all_versions if v.run_id == best_run_id]

if matching:
    best_version = matching[0].version
    print(f'Versão no registry: v{best_version}')

    client.transition_model_version_stage(
        name=REGISTERED_MODEL,
        version=best_version,
        stage='Production',
        archive_existing_versions=True,
    )
    client.update_model_version(
        name=REGISTERED_MODEL,
        version=best_version,
        description=(
            f'{best["model"]} — Acc={best["accuracy"]}, '
            f'F1_macro={best["f1_macro"]}, F1_w={best["f1_weighted"]}. '
            f'SMOTENC + merge_extremes={MERGE_EXTREME_CLASSES}. Run: {best_run_id}'
        )
    )
    print(f'🚀  v{best_version} ({best["model"]}) promovida para Production!')
    print(f'    URI: models:/{REGISTERED_MODEL}/Production')
else:
    print('⚠️  Versão não encontrada automaticamente. Promova manualmente no DagsHub.')

## 11. Resumo Final

In [ ]:
df_final = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('best_estimator',)}
    for r in all_results
]).sort_values('f1_macro', ascending=False)

print('\n' + '='*70)
print('RANKING FINAL — comparação completa no DagsHub:')
print(f'https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO_NAME}.mlflow')
print('='*70)
print(df_final[['model','accuracy','f1_macro','f1_weighted','cv_f1_macro','run_id']]
      .to_string(index=False))
print('\n✅  Todos os experimentos logados.')
print('   No DagsHub → Experiments → selecione múltiplos runs → Compare')